# Indexes in MongoDB

## Overview
MongoDB indexes work similarly to PostgreSQL indexes: they maintain a separate data structure that allows the database to find matching documents without scanning the entire collection.

**Without an index:** MongoDB performs a **COLLSCAN** (collection scan) -- reading every document. On a 10-million-document collection this is slow regardless of how selective the query is.

**With an index:** MongoDB performs an **IXSCAN** -- jumping directly to matching documents.

**The `_id` field is always indexed** (unique). All other fields must be explicitly indexed.

**Key differences from SQL indexes:**
- MongoDB can automatically create **multikey indexes** when a field contains an array -- one index entry per array element
- **TTL indexes** auto-expire documents after a specified time (no SQL equivalent without triggers)
- **Text indexes** provide relevance-scored full-text search
- One collection can have only **one text index** (vs PostgreSQL's multiple GIN indexes)
- `explain()` replaces PostgreSQL's `EXPLAIN ANALYZE` for query plan inspection

---

In [1]:

print("=== Index types in MongoDB ===")
types = [
    ("Single-field",  "Index on one field",                     "patients.create_index('province')"),
    ("Compound",      "Index on multiple fields (leftmost prefix rule applies)",
                      "patients.create_index([('province',1),('last_name',1)])"),
    ("Multikey",      "Auto-created when indexing an array field",
                      "patients.create_index('conditions')"),
    ("Text",          "Full-text search on string fields",      "patients.create_index([('notes','text')])"),
    ("Geospatial 2d", "Flat geometry (legacy lon/lat pairs)",   "collection.create_index([('loc','2d')])"),
    ("Geospatial 2dsphere","Spherical GeoJSON geometries (GPS)", "collection.create_index([('location','2dsphere')])"),
    ("Hashed",        "Equality only; used for hash-based sharding", "collection.create_index([('user_id','hashed')])"),
    ("TTL",           "Auto-expire documents after N seconds",  "logs.create_index('created_at', expireAfterSeconds=86400)"),
    ("Wildcard",      "Index all fields in a subdocument",      "collection.create_index({'metadata.$**': 1})"),
    ("Partial",       "Index only documents matching a filter",  "patients.create_index('province', partialFilterExpression={'active': True})"),
]
print(f"  {'Type':<18s}  {'Purpose':<48s}  Example")
print("  " + "-"*100)
for t, purpose, example in types:
    print(f"  {t:<18s}  {purpose:<48s}  {example}")


=== Index types in MongoDB ===
  Type                Purpose                                           Example
  ----------------------------------------------------------------------------------------------------
  Single-field        Index on one field                                patients.create_index('province')
  Compound            Index on multiple fields (leftmost prefix rule applies)  patients.create_index([('province',1),('last_name',1)])
  Multikey            Auto-created when indexing an array field         patients.create_index('conditions')
  Text                Full-text search on string fields                 patients.create_index([('notes','text')])
  Geospatial 2d       Flat geometry (legacy lon/lat pairs)              collection.create_index([('loc','2d')])
  Geospatial 2dsphere  Spherical GeoJSON geometries (GPS)                collection.create_index([('location','2dsphere')])
  Hashed              Equality only; used for hash-based sharding       collection.crea

---
## Creating and managing indexes

In [2]:

print("=== Creating and managing indexes ===")
index_code = '''
# ── Single-field index ───────────────────────────────────────────
patients.create_index("province")
patients.create_index("patient_id", unique=True)   # unique constraint

# ── Compound index ───────────────────────────────────────────────
# Index direction: 1=ASC, -1=DESC
# Supports queries on: (province), (province + last_name), but NOT (last_name) alone
patients.create_index([("province", 1), ("last_name", 1)])

# ── TTL index: auto-delete session documents after 24 hours ─────
db.sessions.create_index("created_at", expireAfterSeconds=86400)

# ── Text index: full-text search on notes field ──────────────────
encounters.create_index([("notes", pymongo.TEXT)])
# Search:
encounters.find({"$text": {"$search": "hypertension medication"}})
# Relevance score:
encounters.find(
    {"$text": {"$search": "hypertension"}},
    {"score": {"$meta": "textScore"}}
).sort([("score", {"$meta": "textScore"})])

# ── Partial index: only index active patients ────────────────────
patients.create_index(
    "province",
    partialFilterExpression={"active": {"$eq": True}}
)

# ── List all indexes ─────────────────────────────────────────────
for idx in patients.list_indexes():
    print(idx)

# ── Drop index ───────────────────────────────────────────────────
patients.drop_index("province_1")          # by index name
patients.drop_index([("province", 1)])     # by key spec
patients.drop_indexes()                    # drop ALL indexes (except _id)
'''
print(index_code)


=== Creating and managing indexes ===

# ── Single-field index ───────────────────────────────────────────
patients.create_index("province")
patients.create_index("patient_id", unique=True)   # unique constraint

# ── Compound index ───────────────────────────────────────────────
# Index direction: 1=ASC, -1=DESC
# Supports queries on: (province), (province + last_name), but NOT (last_name) alone
patients.create_index([("province", 1), ("last_name", 1)])

# ── TTL index: auto-delete session documents after 24 hours ─────
db.sessions.create_index("created_at", expireAfterSeconds=86400)

# ── Text index: full-text search on notes field ──────────────────
encounters.create_index([("notes", pymongo.TEXT)])
# Search:
encounters.find({"$text": {"$search": "hypertension medication"}})
# Relevance score:
encounters.find(
    {"$text": {"$search": "hypertension"}},
    {"score": {"$meta": "textScore"}}
).sort([("score", {"$meta": "textScore"})])

# ── Partial index: only index active patients ─

---
## explain(): inspecting query plans

In [3]:

print("=== explain() in MongoDB: checking index usage ===")
explain_code = '''
# Check whether a query uses an index:
explanation = patients.find({"province": "NB"}).explain("executionStats")

# Key fields to read:
# explanation["queryPlanner"]["winningPlan"]["inputStage"]["stage"]
#   COLLSCAN  = full collection scan (bad for large collections)
#   IXSCAN    = index scan (good)
#   FETCH     = fetch documents by _id after index scan
#   IDHACK    = direct _id lookup (fastest)

# explanation["executionStats"]
#   nReturned:       actual documents returned
#   totalDocsExamined: documents scanned (should be close to nReturned for a good index)
#   totalKeysExamined: index keys scanned
#   executionTimeMillis: query time in ms

# Ideal: totalDocsExamined == nReturned (index is perfectly selective)
# Bad:   totalDocsExamined >> nReturned (scanning many docs to find few)
'''
print(explain_code)

print("Index efficiency checklist:")
checklist = [
    ("COLLSCAN on large collection",   "Add an index on the filter field"),
    ("totalDocsExamined >> nReturned", "Index exists but not selective enough; consider compound index"),
    ("IXSCAN but slow",                "Index scan + many FETCH ops; add INCLUDE fields or use covered query"),
    ("Sort stage in winning plan",     "Add index that matches the sort order to avoid in-memory sort"),
    ("Multiple indexes for one query", "MongoDB picks one; compound index may be more efficient"),
    ("_id queries are always fast",    "IDHACK uses the implicit unique _id index"),
]
print(f"  {'Signal':<42s}  Action")
print("  " + "-"*75)
for signal, action in checklist:
    print(f"  {signal:<42s}  {action}")


=== explain() in MongoDB: checking index usage ===

# Check whether a query uses an index:
explanation = patients.find({"province": "NB"}).explain("executionStats")

# Key fields to read:
# explanation["queryPlanner"]["winningPlan"]["inputStage"]["stage"]
#   COLLSCAN  = full collection scan (bad for large collections)
#   IXSCAN    = index scan (good)
#   FETCH     = fetch documents by _id after index scan
#   IDHACK    = direct _id lookup (fastest)

# explanation["executionStats"]
#   nReturned:       actual documents returned
#   totalDocsExamined: documents scanned (should be close to nReturned for a good index)
#   totalKeysExamined: index keys scanned
#   executionTimeMillis: query time in ms

# Ideal: totalDocsExamined == nReturned (index is perfectly selective)
# Bad:   totalDocsExamined >> nReturned (scanning many docs to find few)

Index efficiency checklist:
  Signal                                      Action
  ---------------------------------------------------------------

---
## Common Pitfalls

**1. Too many indexes slowing down writes**
Every index must be updated on every insert, update, and delete. A collection with 15 indexes has 15 index updates per write. For write-heavy collections (event logs, sensor data), keep indexes minimal and targeted. Use partial indexes to index only the subset of documents you actually query.

**2. Multikey index on array + compound index limitations**
A compound index cannot have more than one multikey (array) field. `create_index([('conditions', 1), ('diagnoses', 1)])` fails if both fields are arrays. Index each array field separately, or restructure to embed one array.

**3. Not understanding the compound index leftmost prefix rule**
An index on `(province, last_name, dob)` supports queries on `province`, `province + last_name`, and `province + last_name + dob`. It does NOT support queries on `last_name` or `dob` alone, or on `last_name + dob` without `province`. The same rule applies as in PostgreSQL.

**4. TTL index not running immediately**
MongoDB's TTL background thread runs every 60 seconds by default. Documents do not expire at the exact second they hit the TTL threshold -- they may persist for up to 60 seconds longer. Do not use TTL as a precise real-time expiration mechanism.

**5. Text index allowing only one per collection**
A collection can have only one text index. If you need full-text search across multiple fields, define them all in a single text index: `create_index([('title', 'text'), ('notes', 'text'), ('tags', 'text')])`.

**6. drop_indexes() removes all performance tuning**
`drop_indexes()` removes every index except `_id` -- including unique constraints. Dropping all indexes on a production collection causes immediate performance degradation on every query. Always drop indexes individually by name using `drop_index('index_name')`.


---
*sql_methods_library - Samantha McGarrigle*